# Colab Simple Single-Threaded Docling Ingestion (JSONL Optimized)
This notebook guarantees zero Google Drive FUSE crashes by using buffered streaming for reads, and drastically speeds up I/O by outputting a single `.jsonl` file instead of thousands of individual JSONs.

In [ ]:
# 1. Install dependencies
!pip install -q docling pandas seaborn matplotlib

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuration & Initialization
import time
import json
import shutil
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

# --- DRIVE PATHS ---
DRIVE_SOURCE_DIR = Path("/content/drive/MyDrive/Projects/nlp_ml_gcs_pdfs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Projects/output")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- LOCAL PATHS (Colab SSD) ---
LOCAL_SOURCE_DIR = Path("/content/local_pdfs")
LOCAL_OUTPUT_FILE = Path("/content/local_output_batch.jsonl")

# --- OPTIONS ---
MAX_FILES = 100       # Set to None to process all
DISABLE_OCR = True    # Keep True for speed

print("Initializing Docling Model...")
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = not DISABLE_OCR
pipeline_options.do_table_structure = True

converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF],
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
print("Model Initialized successfully!")

In [ ]:
# 4. FUSE-Safe File Copy to Local Storage
def safe_buffered_copy(src: Path, dst: Path):
    with open(src, 'rb') as fsrc:
        with open(dst, 'wb') as fdst:
            while True:
                buf = fsrc.read(1024 * 1024) # 1 MB buffer
                if not buf:
                    break
                fdst.write(buf)

if not DRIVE_SOURCE_DIR.exists():
    print(f"ERROR: Directory {DRIVE_SOURCE_DIR} not found. Check your mount!")
else:
    if LOCAL_SOURCE_DIR.exists(): shutil.rmtree(LOCAL_SOURCE_DIR)
    if LOCAL_OUTPUT_FILE.exists(): LOCAL_OUTPUT_FILE.unlink()
    
    LOCAL_SOURCE_DIR.mkdir(parents=True)
    
    drive_pdfs = list(DRIVE_SOURCE_DIR.glob("*.pdf"))
    if MAX_FILES is not None:
        drive_pdfs = drive_pdfs[:MAX_FILES]
        
    print(f"Copying {len(drive_pdfs)} PDFs using FUSE-safe streaming...")
    for pdf in tqdm(drive_pdfs, desc="Copying files"):
        safe_buffered_copy(pdf, LOCAL_SOURCE_DIR / pdf.name)
        
    print("Copy complete! Local processing will now begin.")

In [ ]:
# 5. Processing Loop (JSONL Output)
local_pdfs = list(LOCAL_SOURCE_DIR.glob("*.pdf"))
metrics = []
start_total = time.perf_counter()

# Open the local JSONL file for appending
with open(LOCAL_OUTPUT_FILE, "a") as out_file:
    for pdf_path in tqdm(local_pdfs, desc="Parsing PDFs"):
        file_size_kb = pdf_path.stat().st_size / 1024
        start_time = time.perf_counter()
        status = "SUCCESS"
        error_msg = ""
        
        try:
            result = converter.convert(pdf_path)
            doc_dict = result.document.export_to_dict()
            
            # Add the filename to the dict so we can identify it later
            doc_dict['_source_filename'] = pdf_path.name
            
            # Write a single line of JSON to the local file
            out_file.write(json.dumps(doc_dict) + "\n")
                
        except Exception as e:
            status = "FAILED"
            error_msg = str(e)
            
        processing_time = time.perf_counter() - start_time
        
        metrics.append({
            "filename": pdf_path.name,
            "file_size_kb": file_size_kb,
            "processing_time_sec": processing_time,
            "status": status,
            "error_msg": error_msg
        })

total_time = time.perf_counter() - start_total

print(f"\nPipeline completed in {total_time:.2f} seconds.")

# Copy the single massive JSONL file safely to Google Drive
if LOCAL_OUTPUT_FILE.exists():
    print("Copying final JSONL batch to Google Drive...")
    final_drive_path = DRIVE_OUTPUT_DIR / f"docling_batch_{int(time.time())}.jsonl"
    safe_buffered_copy(LOCAL_OUTPUT_FILE, final_drive_path)
    print(f"Successfully saved batch to: {final_drive_path}")

df_metrics = pd.DataFrame(metrics)

In [ ]:
# 6. Analytics & Summary
import seaborn as sns
import matplotlib.pyplot as plt

if 'df_metrics' in locals() and not df_metrics.empty:
    success_mask = df_metrics['status'] == 'SUCCESS'
    success_rate = success_mask.mean() * 100
    avg_time = df_metrics.loc[success_mask, 'processing_time_sec'].mean() if success_mask.any() else 0
    throughput = len(df_metrics) / total_time

    print(f"=== INGESTION SUMMARY ===")
    print(f"Total PDFs Processed: {len(df_metrics)}")
    print(f"Success Rate:         {success_rate:.2f}%")
    print(f"Total Wall Time:      {total_time:.2f} s")
    print(f"Throughput:           {throughput:.2f} PDFs / sec")
    print(f"Avg Time per PDF:     {avg_time:.2f} s")

    failures = df_metrics[df_metrics['status'] == 'FAILED']
    if not failures.empty:
        print(f"\nEncountered {len(failures)} failures:")
        display(failures[['filename', 'error_msg']].head())

    if success_mask.any():
        plt.figure(figsize=(10, 5))
        sns.histplot(data=df_metrics[success_mask], x='processing_time_sec', bins=30, kde=True)
        plt.title('Distribution of Docling Processing Times per PDF')
        plt.xlabel('Processing Time (seconds)')
        plt.ylabel('Count')
        plt.show()
else:
    print("No data to analyze.")